In [ ]:
# https://colab.research.google.com/drive/1nzdTOey1S8aHnFUxaRc-8d9zI8cEpo5R#scrollTo=FtViIAfTizDJ

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow import keras
from tensorflow.keras.layers import Dense, Flatten, Reshape, Input, Lambda, BatchNormalization, Dropout
from tensorflow.keras.layers import concatenate

hidden_dim = 2
num_classes = 10
batch_size = 100  # должно быть кратно 60 000 и 10 0000

(x_train, y_train), (x_test, y_test) = mnist.load_data()

# стандартизация входных данных
x_train = x_train / 255
x_test = x_test / 255

x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test = np.reshape(x_test, (len(x_test), 28, 28, 1))

y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)


def dropout_and_batch(x):
    return Dropout(0.3)(BatchNormalization()(x))

input_img = Input(shape=(28, 28, 1))
fl = Flatten()(input_img)
lb = Input(shape=(num_classes,))
x = concatenate([fl, lb])
x = Dense(256, activation='relu')(x)
x = dropout_and_batch(x)
x = Dense(128, activation='relu')(x)
x = dropout_and_batch(x)

z_mean2 = Dense(hidden_dim)(x)
z_log_var = Dense(hidden_dim)(x)

def noiser(args):
    global z_mean, z_log_var
    z_mean, z_log_var = args
    N = tf.random.normal(shape=(batch_size, hidden_dim), mean=0., stddev=1.0)
    return tf.exp(z_log_var / 2) * N + z_mean

h = Lambda(noiser, output_shape=(hidden_dim,))([z_mean2, z_log_var])

input_dec = Input(shape=(hidden_dim,))
lb_dec = Input(shape=(num_classes,))
d = concatenate([input_dec, lb_dec])
d = Dense(128, activation='elu')(d)
d = dropout_and_batch(d)
d = Dense(256, activation='elu')(d)
d = dropout_and_batch(d)
d = Dense(28*28, activation='sigmoid')(d)
decoded = Reshape((28, 28, 1))(d)

encoder = keras.Model([input_img, lb], h, name='encoder')
decoder = keras.Model([input_dec, lb_dec], decoded, name='decoder')
cvae = keras.Model([input_img, lb, lb_dec], decoder([encoder([input_img, lb]), lb_dec]), name="cvae")

z_meaner = keras.Model([input_img, lb], z_mean2)
tr_style = keras.Model([input_img, lb, lb_dec], decoder([z_meaner([input_img, lb]), lb_dec]), name='tr_style')

def vae_loss(x, y):
    x = tf.reshape(x, shape=(batch_size, 28*28))
    y = tf.reshape(y, shape=(batch_size, 28*28))
    loss = tf.reduce_sum(tf.square(x-y), axis=-1)
    kl_loss = -0.5 * tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=-1)
    return (loss + kl_loss)/2/28/28

cvae.compile(optimizer='adam', loss=vae_loss)

cvae.fit([x_train, y_train_cat, y_train_cat], x_train, epochs=5, batch_size=batch_size, shuffle=True)

lb = lb_dec = y_test_cat
h = encoder.predict([x_test, lb], batch_size=batch_size)
plt.scatter(h[:, 0], h[:, 1])


n = 4
total = 2*n+1
input_lbl = np.zeros((1, num_classes))
input_lbl[0, 5] = 1

plt.figure(figsize=(total, total))

h = np.zeros((1, hidden_dim))
num = 1
for i in range(-n, n+1):
    for j in range(-n, n+1):
        ax = plt.subplot(total, total, num)
        num += 1
        h[0, :] = [1*i/n, 1*j/n]
        img = decoder.predict([h, input_lbl])
        plt.imshow(img.squeeze(), cmap='gray')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)


def plot_digits(*images):
    images = [x.squeeze() for x in images]
    n = min([x.shape[0] for x in images])

    plt.figure(figsize=(n, len(images)))
    for j in range(n):
        for i in range(len(images)):
            ax = plt.subplot(len(images), n, i*n + j + 1)
            plt.imshow(images[i][j])
            plt.gray()
            ax.get_xaxis().set_visible(False)
            ax.get_yaxis().set_visible(False)

    plt.show()

dig1 = 5
dig2 = 2

num = 10
X = x_train[y_train == dig1][:num]

lb_1 = np.zeros((num, num_classes))
lb_1[:, dig1] = 1

plot_digits(X)

for i in range(num_classes):
    lb_2 = np.zeros((num, num_classes))
    lb_2[:, i] = 1

    Y = tr_style.predict([X, lb_1, lb_2], batch_size=num)
    plot_digits(Y)

## Izah

In [ ]:
# https://colab.research.google.com/drive/1nzdTOey1S8aHnFUxaRc-8d9zI8cEpo5R#scrollTo=FtViIAfTizDJ

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
# TensorFlow-un texniki xəbərdarlıqlarını (info/warning mesajlarını) gizlədir.
#   '2' → INFO və WARNING gizlədilir, yalnız ERROR göstərilir.
# VACİB: bu sətir tensorflow import olunmamışdan ƏVVƏL yazılmalıdır, əks halda
# təsir etmir. Burada düzgün yerdədir (aşağıdakı import-lardan əvvəl).
# Kodun işləməsinə təsir etmir, yalnız konsolu təmiz saxlayır.

import numpy as np
# NumPy — massivlərlə işləmək üçün.
# Bu kodda:
# - np.reshape → massivin formasını dəyişmək (28x28 → 28x28x1)
# - np.zeros   → sıfırlardan ibarət massiv (one-hot etiketlər üçün)

import matplotlib.pyplot as plt
# matplotlib — şəkilləri və qrafikləri göstərmək üçün.
# Bu kodda:
# - plt.scatter  → nöqtə buludu (gizli fəzanın xəritəsi)
# - plt.figure   → yeni qrafik pəncərəsi açmaq
# - plt.subplot  → bir pəncərədə şəkil şəbəkəsi (cədvəl)
# - plt.imshow   → massivi şəkil kimi göstərmək
# - plt.show     → qrafikləri ekrana çıxarmaq

import tensorflow as tf
# TensorFlow-un özü. Xüsusi əməliyyatlar üçün lazımdır:
#   tf.random.normal  → təsadüfi səs-küy
#   tf.exp, tf.square → eksponent, kvadrat
#   tf.reduce_sum     → cəmləmək
#   tf.reshape        → forma dəyişmək
# (Əvvəlki versiyada bunlar K.xxx idi, indi tf.xxx ilə əvəz olunub ki, xəta çıxmasın.)

from tensorflow.keras.datasets import mnist
# MNIST — əl ilə yazılmış rəqəmlər (0..9).
#   - 60000 öyrətmə şəkli, 10000 test şəkli
#   - hər şəkil 28x28 piksel, boz tonlu, piksel qiyməti 0..255

from tensorflow import keras
# Keras-ın özü. keras.Model(...) və keras.utils.to_categorical(...) üçün lazımdır.

from tensorflow.keras.layers import Dense, Flatten, Reshape, Input, Lambda, BatchNormalization, Dropout
# Dense              → tam bağlı qat
# Flatten            → (28, 28, 1) → (784,)
# Reshape            → (784,) → (28, 28, 1)
# Input              → modelin giriş formasını elan edir
# Lambda             → öz yazdığımız funksiyanı "qat" kimi şəbəkəyə qoşur
#                      (burada: səs-küy əlavə edən noiser)
# BatchNormalization → hər batch-də qatın çıxışını normallaşdırır, öyrənməni sabitləşdirir
# Dropout            → öyrətmə zamanı neyronların bir hissəsini təsadüfi söndürür,
#                      əzbərləməyə (overfitting) qarşı

from tensorflow.keras.layers import concatenate
# concatenate → bir neçə tensoru bir-birinin ardınca BİRLƏŞDİRİR.
# YENİDİR: əvvəlki VAE-də yox idi. Burada şəkil vektorunu və rəqəm
# etiketini birləşdirmək üçün lazımdır.
# Nümunə: (784,) və (10,) → (794,)


# =====================================================================
# BU KOD NƏDİR? — CONDITIONAL VAE (CVAE, "şərti variasiyalı avtoenkoder")
# =====================================================================
#
# Əvvəlki VAE-dən ƏSAS FƏRQ: modelə şəklin yanında həm də ETİKET
# (bu hansı rəqəmdir: 0, 1, ..., 9) verilir. Yəni model "şərtlə" işləyir:
# "bu rəqəm 5-dir, onu necə çəkək?"
#
#   Adi VAE:  şəkil              ──► encoder ──► z ──► decoder          ──► şəkil
#   CVAE:     şəkil + ETİKET     ──► encoder ──► z ──► decoder + ETİKET ──► şəkil
#
# NİYƏ BU BELƏ GÜCLÜDÜR?
#   Adi VAE-də gizli fəza (2 ədəd) iki şeyi eyni anda saxlamalı idi:
#     (a) bu hansı rəqəmdir,
#     (b) necə yazılıb (qalınlıq, əyiklik, incəlik...).
#   CVAE-də (a) artıq ETİKETLƏ verilir. Ona görə gizli fəzadakı 2 ədəd
#   yalnız (b)-ni, yəni YAZI ÜSLUBUNU (style) öyrənir.
#   Nəticə: eyni z-ni fərqli etiketlə decoder-ə versək, EYNİ üslubda
#   FƏRQLİ rəqəm alınır. Kodun sonundakı "style transfer" (üslub köçürmə)
#   məhz buna əsaslanır.
#
# ƏVVƏLKİ VAE İLƏ MÜQAYİSƏ:
#
#                          Əvvəlki VAE            Bu kod (CVAE)
# Encoder girişi:          şəkil                  şəkil + etiket (794 ədəd)
# Decoder girişi:          z (2 ədəd)             z + etiket (12 ədəd)
# Model girişləri:         1 (şəkil)              3 (şəkil, etiket, dekoder etiketi)
# Gizli fəza nəyi saxlayır: rəqəm + üslub         yalnız üslub
# Scatter-də qruplar:      rəqəmlərə görə qruplar rəqəmlərə görə QRUP YOXDUR
# Decoder aktivasiyası:    relu                   elu
# batch_size:              60                     100
# Loss miqyası:            cəm (onlarla)          cəm / (2·28·28) (təxminən 0.0x)
# Grid aralığı:            [-3, 3]                [-1, 1]
# Əlavə model:             yox                    z_meaner və tr_style
#
#   şəkil (784) + etiket (10) ─► [ENCODER] ─► z_mean, z_log_var ─► (səs-küy) ─► h (2)
#                                                                                 │
#                                       etiket (10) ─────────────────────┐       │
#                                                                         ▼       ▼
#                                                                       [DECODER] ─► şəkil (784)


# ---------------------------------------------------------------------
# 1) PARAMETRLƏR
# ---------------------------------------------------------------------

hidden_dim = 2
# Gizli fəzanın ölçüsü = 2 (müstəvidə nöqtə kimi görmək üçün).
# Burada bu 2 ədəd "yazı üslubunu" ifadə edir.

num_classes = 10
# Sinif sayı: rəqəmlər 0..9.

batch_size = 100  # должно быть кратно 60 000 и 10 0000
# (Rusca şərh: "60 000-ə və 10 000-ə bölünməlidir"; "10 0000" yazı səhvidir, 10 000 olmalıdır.)
# Hər addımda 100 şəkil. 60000 / 100 = 600 addım hər epoch-da, qalıq YOXDUR.
# 10000 / 100 = 100, test üçün də qalıq yoxdur.
#
# NİYƏ BÖLÜNMƏLİDİR? noiser və vae_loss funksiyalarında batch ölçüsü SABİT
# yazılıb: (batch_size, ...). Sonuncu batch yarımçıq olsaydı, forma uyğun
# gəlməzdi və xəta çıxardı.


# ---------------------------------------------------------------------
# 2) DATANIN YÜKLƏNMƏSİ VƏ HAZIRLANMASI
# ---------------------------------------------------------------------

(x_train, y_train), (x_test, y_test) = mnist.load_data()
# Dörd massiv:
#   x_train → (60000, 28, 28), uint8 (0..255)
#   y_train → (60000,) rəqəmlər (0..9). BU KODDA İSTİFADƏ OLUNUR (etiket kimi)!
#   x_test  → (10000, 28, 28)
#   y_test  → (10000,)

# стандартизация входных данных
x_train = x_train / 255
x_test = x_test / 255
# (Rusca şərh: "giriş datasının standartlaşdırılması".)
# Piksellər 0..255 → 0..1. Səbəb: şəbəkə kiçik ədədlərlə yaxşı öyrənir və
# decoder-in sigmoid çıxışı da 0..1 aralığındadır.

x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test = np.reshape(x_test, (len(x_test), 28, 28, 1))
# Kanal ölçüsü əlavə edir:
#   (60000, 28, 28) → (60000, 28, 28, 1)
#   (10000, 28, 28) → (10000, 28, 28, 1)
# Son 1 → tək kanal (boz ton).

y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)
# ONE-HOT KODLAŞDIRMA: rəqəmi 10 uzunluqlu vektora çevirir.
#   5 → [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
#   2 → [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
# Formalar: y_train_cat → (60000, 10), y_test_cat → (10000, 10).
# NİYƏ ELƏ? "5" rəqəmini şəbəkəyə birbaşa ədəd kimi versək, şəbəkə elə
# düşünər ki, 5 "2-dən böyükdür" və ya "4-ə yaxındır". Etiketlər isə
# sadəcə kateqoriyalardır. One-hot hər kateqoriyanı ayrı "açar" edir.


# ---------------------------------------------------------------------
# 3) KÖMƏKÇİ FUNKSİYA
# ---------------------------------------------------------------------

def dropout_and_batch(x):
    return Dropout(0.3)(BatchNormalization()(x))
# Yazını qısaltmaq üçün: x-i əvvəl BatchNormalization-dan, sonra Dropout-dan
# keçirir. Sıra: Dense → BatchNorm → Dropout.
#
# BatchNormalization: hər neyronun çıxışını batch üzrə normallaşdırır.
#   Parametr: hər neyron üçün 4 ədəd (2-si öyrənilir, 2-si statistika).
# Dropout(0.3): öyrətmə zamanı neyronların 30%-ni təsadüfi 0 edir.
#   Parametr: yoxdur. predict zamanı AVTOMATİK söndürülür.
# Hər çağırışda YENİ qatlar yaradılır (qatlar paylaşılmır).


# ---------------------------------------------------------------------
# 4) ENCODER HİSSƏSİ
# ---------------------------------------------------------------------
#
#   şəkil (28,28,1) ─► Flatten ─► (784,) ──┐
#                                            ├─► concatenate ─► (794,)
#   etiket (10,) ───────────────────────────┘
#
#   (794,) ─► Dense+BN+Drop (256,) ─► Dense+BN+Drop (128,) ─┬─► z_mean2   (2,)
#                                                             └─► z_log_var (2,)
#                                                                   │
#                                                                   ▼
#                                                     noiser (Lambda) ─► h (2,)

input_img = Input(shape=(28, 28, 1))
# Şəkil girişi: forma (28, 28, 1). Batch ölçüsü yazılmır.

fl = Flatten()(input_img)
# (28, 28, 1) → (784,). Parametr: yoxdur.

lb = Input(shape=(num_classes,))
# ETİKET girişi: forma (10,) one-hot vektor. Bu, ikinci giriş yeridir.
# DİQQƏT: lb sonradan (aşağıda) numpy massivi ilə eyni adla əvəz ediləcək.

x = concatenate([fl, lb])
# Şəkil vektoru (784) + etiket vektoru (10) → (794,).
# Beləcə Dense qat həm piksellərə, həm də "bu hansı rəqəmdir" məlumatına
# baxır. DİQQƏT: bu x şəbəkənin cari nöqtəsidir, x_train ilə qarışdırmayın.

x = Dense(256, activation='relu')(x)
# 794 → 256, ReLU. Parametr: 794 × 256 + 256 = 203520.

x = dropout_and_batch(x)
# BatchNorm: 4 × 256 = 1024 parametr.

x = Dense(128, activation='relu')(x)
# 256 → 128, ReLU. Parametr: 256 × 128 + 128 = 32896.

x = dropout_and_batch(x)
# BatchNorm: 4 × 128 = 512 parametr.

z_mean2 = Dense(hidden_dim)(x)
# 128 → 2, aktivasiya YOXDUR. Parametr: 128 × 2 + 2 = 258.
# Mənası: şəklin gizli fəzadakı "buludunun" MƏRKƏZİ (üslub koordinatları).
#
# NİYƏ ADI z_mean2? Aşağıdakı noiser funksiyası global z_mean adlı
# dəyişən yaradır (loss üçün). z_mean2 isə bu qatın çıxışının AYRICA adıdır;
# ondan z_meaner modelini qurmaq üçün istifadə edəcəyik. Hər ikisi eyni
# tensora işarə edir (aşağıya baxın), sadəcə fərqli adlardır.

z_log_var = Dense(hidden_dim)(x)
# 128 → 2, aktivasiya YOXDUR. Parametr: 258.
# Mənası: buludun yayılmasının LOQARİFMİ (log σ²).
# Niyə log? σ² həmişə müsbət olmalıdır; log isə istənilən ədəd ola bilər,
# exp(log σ²) isə həmişə müsbətdir.


def noiser(args):
    global z_mean, z_log_var
    z_mean, z_log_var = args
    N = tf.random.normal(shape=(batch_size, hidden_dim), mean=0., stddev=1.0)
    return tf.exp(z_log_var / 2) * N + z_mean
# "REPARAMETRİZASİYA HİLƏSİ" (əvvəlki VAE-dəki kimi).
#
#   args → [z_mean2, z_log_var] siyahısı (Lambda qatı belə verir).
#
#   global z_mean, z_log_var:
#     Funksiya daxilində bu iki adı GLOBAL dəyişənlərə mənimsədir, ki
#     aşağıdakı vae_loss bunlara çata bilsin (KL hissəsi üçün lazımdır).
#     Nəticədə global z_mean = z_mean2 (eyni tensor). z_log_var isə əvvəlcədən
#     eyni adla mövcud idi və eyni tensora yenidən mənimsədilir.
#     Bu, loss-a məlumat ötürmək üçün "qısa yol"dur: işləyir, amma təmiz
#     deyil və bir qədər kövrəkdir (gizli yan təsir). Təmiz variant: xüsusi
#     Layer yazıb add_loss istifadə etmək.
#
#   N = tf.random.normal(...):
#     Forma (100, 2): hər şəkil üçün 2 təsadüfi ədəd, orta 0, σ = 1.
#
#   return exp(z_log_var / 2) * N + z_mean:
#     exp(log σ² / 2) = σ. Yəni h = σ · N + μ. Bu, μ mərkəzli, σ yayılmalı
#     normal paylanmadan təsadüfi nöqtə seçməkdir. Təsadüfiliyi N-ə köçürmək
#     sayəsində gradient μ və σ-ya çatır (şəbəkə öyrənə bilir).
#
# DİQQƏT: Lambda qat öyrətmə və predict zamanı EYNİ işləyir, yəni encoder
# həmişə SƏS-KÜYLÜ çıxış verir.

h = Lambda(noiser, output_shape=(hidden_dim,))([z_mean2, z_log_var])
# Lambda qatı noiser-i şəbəkəyə qoşur. output_shape=(2,) → çıxış 2 ədəddir.
# h: hər şəkil üçün gizli fəzada təsadüfi seçilmiş nöqtə, forma (batch, 2).
# Parametr: yoxdur.
# DİQQƏT: h burada simvolik tensordur. Aşağıda h adı iki dəfə də numpy
# massivi üçün yenidən istifadə olunur.


# ---------------------------------------------------------------------
# 5) DECODER HİSSƏSİ
# ---------------------------------------------------------------------
#
#   z (2,) ───────────┐
#                      ├─► concatenate ─► (12,)
#   etiket (10,) ─────┘
#
#   (12,) ─► Dense+BN+Drop (128,) ─► Dense+BN+Drop (256,) ─► Dense (784,) ─► Reshape (28,28,1)

input_dec = Input(shape=(hidden_dim,))
# Decoder-in 1-ci girişi: 2 ədədlik gizli kod z. Encoder-dən ASILI DEYİL,
# istənilən 2 ədədi birbaşa verə bilərik (aşağıdakı şəbəkədə belə edirik).

lb_dec = Input(shape=(num_classes,))
# Decoder-in 2-ci girişi: ETİKET (one-hot, 10). "Hansı rəqəmi çəkək?"
#
# NİYƏ AYRICA lb_dec, lb-dən istifadə etmirik? Çünki bunlar fərqli şeylərdir:
#   lb     → encoder-ə deyir: "bu şəkil hansı rəqəmdir".
#   lb_dec → decoder-ə deyir: "hansı rəqəmi QURMAQ lazımdır".
# Öyrətmədə eynidirlər, amma test zamanı fərqli ola bilər. Məhz bu fərq
# "üslub köçürməyə" imkan verir (1 → 5-in üslubunda 2 çək).

d = concatenate([input_dec, lb_dec])
# z (2) + etiket (10) → (12,).

d = Dense(128, activation='elu')(d)
# 12 → 128, ELU. Parametr: 12 × 128 + 128 = 1664.
# ELU nədir? ReLU kimidir, amma mənfi tərəfdə 0 yox, hamar əyri verir.
# Üstünlüyü: "ölü neyron" riski azdır, öyrənmə bir az hamar gedir.

d = dropout_and_batch(d)
# BatchNorm: 512 parametr.

d = Dense(256, activation='elu')(d)
# 128 → 256, ELU. Parametr: 128 × 256 + 256 = 33024.

d = dropout_and_batch(d)
# BatchNorm: 1024 parametr.

d = Dense(28*28, activation='sigmoid')(d)
# 256 → 784, sigmoid (çıxış 0..1, piksel aralığı ilə eyni).
# Parametr: 256 × 784 + 784 = 201488.

decoded = Reshape((28, 28, 1))(d)
# (784,) → (28, 28, 1). Parametr: yoxdur.


# ---------------------------------------------------------------------
# 6) MODELLƏRİN YIĞILMASI
# ---------------------------------------------------------------------

encoder = keras.Model([input_img, lb], h, name='encoder')
# Girişlər: [şəkil, etiket]. Çıxış: h (2 ədəd, səs-küylü).
# Parametr: 238468 (203520 + 1024 + 32896 + 512 + 258 + 258).
# İstifadə: encoder.predict([şəkillər, etiketlər]) → forma (sayı, 2).

decoder = keras.Model([input_dec, lb_dec], decoded, name='decoder')
# Girişlər: [z, etiket]. Çıxış: şəkil.
# Parametr: 237712 (1664 + 512 + 33024 + 1024 + 201488).
# İstifadə: decoder.predict([z, etiketlər]) → forma (sayı, 28, 28, 1).

cvae = keras.Model([input_img, lb, lb_dec], decoder([encoder([input_img, lb]), lb_dec]), name="cvae")
# Bütöv model: ÜÇ giriş.
#   input_img → encoder-ə şəkil
#   lb        → encoder-ə etiket
#   lb_dec    → decoder-ə etiket
# Axın: encoder([şəkil, lb]) → h → decoder([h, lb_dec]) → şəkil.
# Ümumi parametr: 238468 + 237712 = 476180.
# Üç model ÇƏKİLƏRİ PAYLAŞIR: cvae-ni öyrətdikdə encoder və decoder də
# öyrənir. Ayrıca fit() çağırmaq lazım deyil.

z_meaner = keras.Model([input_img, lb], z_mean2)
# YENİ MODEL: şəkil və etiket girir, yalnız MƏRKƏZ z_mean2 çıxır
# (səs-küy YOXDUR). Encoder-in qatlarını paylaşır (Lambda-dan əvvəlki hissə).
# NİYƏ LAZIMDIR? Encoder həmişə səs-küylü nəticə verir. Üslub köçürmədə
# isə səs-küy istəmirik: eyni şəkil həmişə eyni z-ni versin.

tr_style = keras.Model([input_img, lb, lb_dec], decoder([z_meaner([input_img, lb]), lb_dec]), name='tr_style')
# "STYLE TRANSFER" (üslub köçürmə) modeli:
#   şəkil + etiket (lb) → z_meaner → səs-küysüz z (üslub) → decoder + lb_dec → şəkil
# Məna: şəklin üslubunu (z) çıxar, sonra onu BAŞQA etiketlə decoder-ə ver.
# Məsələn: bir "5" şəklinin üslubu + etiket "2" → həmin üslubda "2".
# Öyrədilmir (çəkilər cvae ilə paylaşılır, öz fit()-i yoxdur).
#
# BONUS: z_meaner-də noiser olmadığı üçün tr_style ilə predict edərkən
# batch_size=100 məcburi deyil, istənilən ölçüdə işləyir (aşağıda 10 istifadə olunur).


# ---------------------------------------------------------------------
# 7) LOSS FUNKSİYASI
# ---------------------------------------------------------------------

def vae_loss(x, y):
    x = tf.reshape(x, shape=(batch_size, 28*28))
    y = tf.reshape(y, shape=(batch_size, 28*28))
    loss = tf.reduce_sum(tf.square(x-y), axis=-1)
    kl_loss = -0.5 * tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=-1)
    return (loss + kl_loss)/2/28/28
# Loss İKİ hissədən ibarətdir (əvvəlki VAE ilə eyni). Burada x = orijinal
# (y_true), y = bərpa (y_pred). Bu x, y adları funksiyadan kənardakı
# x-dən fərqlidir.
#
# a) BƏRPA XƏTASI: şəkillər (100, 784) formasına düzləşdirilir,
#      loss = Σ (x − y)²  → hər şəkil üçün 784 pikselin kvadrat xətalarının cəmi.
#
# b) KL-DİVERGENSİYA:
#      -0.5 · Σ ( 1 + log σ² − μ² − σ² )
#    Hər şəklin buludunu standart normal N(0, 1)-dən çox uzaqlaşmağa qoymur.
#      - μ² termi → mərkəzləri 0-a çəkir.
#      - σ² − log σ² − 1 termi → yayılmanı 1-ə yaxın saxlayır.
#    z_mean və z_log_var GLOBAL dəyişənlərdir (noiser-də mənimsədilib).
#
# SON BÖLMƏ: (loss + kl_loss)/2/28/28
#   Cəmi 2·28·28 = 1568-ə bölür. Bu, yalnız MİQYASI kiçildir ki, çıxan
#   rəqəmlər onlarla yox, təxminən 0.0x olsun və oxunması rahat olsun.
#   Bərpa və KL arasındakı NİSBƏT dəyişmir, çünki hər ikisi eyni ədədə bölünür.
#   Adam optimizer miqyasa nisbətən həssas olmadığı üçün bu, öyrənməyə
#   əsaslı təsir etmir.
#
# Funksiya hər şəkil üçün forma (100,) vektor qaytarır; Keras ortalama alır.

cvae.compile(optimizer='adam', loss=vae_loss)
# Adam optimizer, defolt öyrənmə sürəti 0.001. Loss — yuxarıdakı xüsusi funksiya.
# metrics yoxdur (accuracy burada mənasızdır).


# ---------------------------------------------------------------------
# 8) ÖYRƏTMƏ
# ---------------------------------------------------------------------

cvae.fit([x_train, y_train_cat, y_train_cat], x_train, epochs=5, batch_size=batch_size, shuffle=True)
# Girişlər (X): [şəkillər, encoder üçün etiketlər, decoder üçün etiketlər].
#   y_train_cat İKİ DƏFƏ verilir, çünki modelin üç girişi var:
#   input_img, lb, lb_dec. Öyrətmədə encoder və decoder eyni (düzgün)
#   etiketi görür.
# Cavab (Y): x_train. Avtoenkoder məntiqi: model şəkli bərpa etməlidir.
#
# epochs=5        → bütün 60000 şəkil 5 dəfə keçilir.
# batch_size=100  → hər addımda 100 şəkil; epoch-da 600 addım, cəmi 3000 yeniləmə.
# shuffle=True    → hər epoch-da sıra qarışdırılır.
#
# NƏ GÖZLƏMƏK LAZIMDIR? Loss miqyas bölməsi sayəsində təxminən 0.0x ətrafında
# olacaq və epoch-lar keçdikcə azalmalıdır. Dəqiq rəqəm hər işlətmədə dəyişir.
# Dropout(0.3) öyrətmədə loss-u bir az yüksək saxlayır.
#
# DİQQƏT: öyrətmədən sonra noiser-in global z_mean/z_log_var dəyişənləri
# predict-in çağırışları zamanı yenidən mənimsədilə bilər. Ona görə fit-i
# predict-dən SONRA yenidən çağırmaq istəsəniz, modelləri baştan qurmaq
# daha etibarlıdır.


# ---------------------------------------------------------------------
# 9) GİZLİ FƏZANIN GÖRÜNTÜSÜ
# ---------------------------------------------------------------------

lb = lb_dec = y_test_cat
# DİQQƏT: lb və lb_dec adları İKİ ŞEYİ eyni anda dəyişir. Əvvəl bunlar
# Keras Input tensorları idi, indi eyni ada test etiketləri massivi
# (10000, 10) mənimsədilir. Modellər artıq qurulub, ona görə bu təhlükəsizdir,
# amma adlar artıq Input DEYİL. Əgər sonra modeli yenidən qurmaq istəsəniz,
# Input-ları yenidən yaratmaq lazım olacaq.
# (lb_dec bu hissədə istifadə olunmur, sadəcə yanaşı mənimsədilib.)

h = encoder.predict([x_test, lb], batch_size=batch_size)
# Bütün 10000 test şəklini encoder-dən keçirir. Forma: (10000, 2).
# batch_size=100 vacibdir: noiser sabit (100, 2) forma tələb edir.
# DİQQƏT: h burada numpy massivdir (yuxarıdakı simvolik tensor yox).
# Bu, səs-küylü çıxışdır (z_mean deyil).

plt.scatter(h[:, 0], h[:, 1])
# 10000 nöqtəni müstəvidə çəkir (x oxu = 1-ci koordinat, y oxu = 2-ci).
# NƏ GÖRMƏK LAZIMDIR? Adi VAE-dən FƏRQLİ olaraq rəqəmlər üzrə ayrı qruplar
# GÖRÜNMƏYƏCƏK. Nöqtələr 0 ətrafında bir bulud kimi qarışacaq. Səbəb:
# "hansı rəqəm" məlumatı artıq etiketlə verilir, ona görə gizli fəzaya
# onu yazmağa ehtiyac qalmır. Burada bulud yalnız üslub dəyişkənliyini
# ifadə edir. Bu pis deyil, əksinə CVAE-nin məqsədinə uyğundur.
# Rəng yoxdur. İstəsəniz y_test ilə rəngləyin:
#   plt.scatter(h[:, 0], h[:, 1], c=y_test, cmap='tab10', s=2)
#   plt.colorbar()
# (Rənglərin qarışdığını görəcəksiniz, bu normaldır.)


# ---------------------------------------------------------------------
# 10) GENERASİYA: ETİKET 5 ÜÇÜN GİZLİ FƏZANIN ŞƏBƏKƏSİ
# ---------------------------------------------------------------------

n = 4
# Hər istiqamətdə 4 addım: -4..4.

total = 2*n+1
# 2·4 + 1 = 9. Şəbəkə 9 × 9 = 81 şəkildən ibarət olacaq.

input_lbl = np.zeros((1, num_classes))
input_lbl[0, 5] = 1
# Etiket vektoru: [0, 0, 0, 0, 0, 1, 0, 0, 0, 0], yəni "5 rəqəmini çək".
# Forma (1, 10), batch ölçüsü 1 ilə.

plt.figure(figsize=(total, total))
# YENİ qrafik pəncərəsi (9×9 düym). scatter ayrı pəncərədə qalır.

h = np.zeros((1, hidden_dim))
# DİQQƏT: h yenidən mənimsədilir (üçüncü dəfə!). İndi h forma (1, 2) olan
# sıfır massivdir. Hər addımda içi doldurulacaq və decoder-ə veriləcək.

num = 1
# plt.subplot nömrəsi 1-dən başlayır (soldan sağa, yuxarıdan aşağı).

for i in range(-n, n+1):
    for j in range(-n, n+1):
        # i, j: -4, -3, ..., 0, ..., 4. Hər cüt gizli fəzada bir nöqtədir.
        # i xarici dövr → SƏTİR (1-ci koordinat), j daxili dövr → SÜTUN (2-ci koordinat).

        ax = plt.subplot(total, total, num)
        # 9×9 şəbəkədə num-cu xananı seçir.

        num += 1

        h[0, :] = [1*i/n, 1*j/n]
        # Nöqtə: [i/4, j/4]. Koordinatlar -1-dən +1-ə qədər 0.25 addımla gedir.
        # Əvvəlki VAE-də aralıq [-3, 3] idi. Burada [-1, 1] seçilib, yəni
        # mərkəzə daha yaxın bölgə nəzərdən keçirilir.

        img = decoder.predict([h, input_lbl])
        # decoder-ə iki giriş: nöqtə h (1, 2) və etiket (1, 10).
        # Çıxış: (1, 28, 28, 1). Hər xanada nöqtə fərqlidir, etiket isə HƏMİŞƏ 5.
        # DİQQƏT: 81 ayrı predict çağırışı yavaşdır. Sürətli yol: 81 nöqtəni
        # bir massivə yığıb bir dəfə çağırmaq.
        # Decoder-də noiser yoxdur, ona görə təsadüfilik yoxdur.

        plt.imshow(img.squeeze(), cmap='gray')
        # .squeeze(): (1, 28, 28, 1) → (28, 28). cmap='gray': boz ton.

        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        # Oxları gizlədir.

# NƏTİCƏ: 9×9 cədvəl, hamısı "5" rəqəmidir, amma ÜSLUB (əyiklik, qalınlıq)
# şəbəkə boyunca hamar dəyişir. Bu, z-nin üslubu saxladığını göstərir.
# Əgər etiketi dəyişsək (məsələn input_lbl[0, 7] = 1), eyni üslub xəritəsi
# "7"-lər üçün alınar.


# ---------------------------------------------------------------------
# 11) KÖMƏKÇİ FUNKSİYA: RƏQƏMLƏRİ SIRA İLƏ GÖSTƏRMƏK
# ---------------------------------------------------------------------

def plot_digits(*images):
    images = [x.squeeze() for x in images]
    n = min([x.shape[0] for x in images])

    plt.figure(figsize=(n, len(images)))
    for j in range(n):
        for i in range(len(images)):
            ax = plt.subplot(len(images), n, i*n + j + 1)
            plt.imshow(images[i][j])
            plt.gray()
            ax.get_xaxis().set_visible(False)
            ax.get_yaxis().set_visible(False)

    plt.show()
# *images → istənilən sayda massiv qəbul edir (hər biri şəkil toplusudur).
#
# images = [x.squeeze() for x in images]
#   Hər toplunun (N, 28, 28, 1) formasını (N, 28, 28) edir.
#   DİQQƏT: topluda yalnız 1 şəkil olsa, squeeze batch ölçüsünü də silər
#   və forma pozular. Burada hər topluda 10 şəkil var, problem yoxdur.
#
# n = min([x.shape[0] for x in images])
#   Hər toplunun şəkil sayının MİNİMUMU. (Bu n lokaldır, yuxarıdakı qlobal
#   n-i dəyişmir.)
#
# plt.figure(figsize=(n, len(images)))
#   Eni n, hündürlüyü topluların sayı qədər pəncərə.
#
# Dövr: j → sütun (hansı şəkil), i → sətir (hansı topluya aid).
#   plt.subplot(len(images), n, i*n + j + 1) → xananın nömrəsi.
#   plt.imshow(images[i][j]) → j-ci şəkli i-ci sətirdə göstərir.
#   plt.gray() → boz rəng palitrası.
#
# plt.show() → pəncərəni ekrana çıxarır.
# DİQQƏT: .py faylında plt.show() ilk dəfə burada çağırılır, ona görə
# əvvəlki qrafiklər (scatter və 9×9 şəbəkə) də bu anda görünəcək.


# ---------------------------------------------------------------------
# 12) ÜSLUB KÖÇÜRMƏ (STYLE TRANSFER)
# ---------------------------------------------------------------------

dig1 = 5
# Mənbə rəqəm: üslubunu götürəcəyimiz rəqəm (5).

dig2 = 2
# Bu dəyişən BU KODDA İSTİFADƏ OLUNMUR. Görünür, əvvəlki sınaqdan qalıb.
# Zərərsizdir, silmək olar.

num = 10
# Neçə şəkil götürəcəyik (10).
# DİQQƏT: num yuxarıda subplot sayğacı idi (81-ə çatmışdı), indi yenidən 10-a mənimsədilir.

X = x_train[y_train == dig1][:num]
# Maska ilə seçmə: y_train == 5 olan bütün şəkillər, onlardan ilk 10-u.
# Forma: (10, 28, 28, 1).

lb_1 = np.zeros((num, num_classes))
lb_1[:, dig1] = 1
# Hər 10 şəkil üçün etiket "5" (one-hot). Forma (10, 10).
# lb_1[:, 5] = 1 → bütün sətirlərdə 5-ci sütun 1 olur.

plot_digits(X)
# Orijinal 10 ədəd "5" şəklini bir sətirdə göstərir (müqayisə üçün).

for i in range(num_classes):
    lb_2 = np.zeros((num, num_classes))
    lb_2[:, i] = 1
    # Hədəf etiket: i rəqəmi (0-dan 9-a qədər). Forma (10, 10).

    Y = tr_style.predict([X, lb_1, lb_2], batch_size=num)
    # tr_style-ə üç giriş:
    #   X    → orijinal "5" şəkilləri (üslubu buradan çıxarır)
    #   lb_1 → onların həqiqi etiketi (5), encoder üçün
    #   lb_2 → decoder üçün HƏDƏF etiket (i)
    # Çıxış: (10, 28, 28, 1), hər biri həmin "5"-in üslubunda çəkilmiş i rəqəmi.
    # batch_size=num=10 işləyir, çünki tr_style noiser-dən keçmir (z_meaner
    # səs-küysüzdür), ona görə sabit 100 ölçüsü tələb olunmur.

    plot_digits(Y)
    # Nəticəni bir sətirdə göstərir.

# NƏTİCƏ: 1 sətir orijinal "5"-lər + 10 sətir (i = 0, 1, ..., 9) bu 5-lərin
# üslubunda çəkilmiş rəqəmlər. Gözləmək olar:
#   - Hər sütunda eyni üslub qalır (əyiklik, qalınlıq), amma rəqəm dəyişir.
#   - i = 5 üçün nəticə orijinal "5"-ə ən yaxın olmalıdır (bulanıq variantı).
#   - Şəkillər bir qədər bulanıq olacaq (2 ölçü + 5 epoch + KL nizamlaması).
#
# QEYDLƏR:
#   1) Üslub z-də yalnız 2 ədədlə saxlanır, ona görə üslub köçürmə kobud olur.
#      Daha zəngin üslub üçün hidden_dim artırıla bilər.
#   2) Hər işlətmədə çəkilər təsadüfi başladığı üçün konkret nəticələr
#      dəyişir, amma ümumi qanunauyğunluq (üslub qalır, rəqəm dəyişir) qalır.

## Mənası

Yaxşı sual. Doğrudan cavab: **label-i özümüz veririksə, bu "proqnoz" deyil, generasiyadır.** Praktikada bunun mənası da məhz budur.

## Praktikada nəyə lazımdır?

**1. Sifarişlə məlumat generasiyası (ən əsas istifadə)**
"Mənə 8 rəqəmi yarat" deyirsiniz, model yeni, əvvəl heç görünməyən bir "8" şəkli çəkir. Klassik model (classifier) şəkli görüb "bu nədir?" deyir; CVAE isə tərsini edir: "bu olsun" deyirsiniz, şəkli yaradır.

**2. Data augmentation**
Bəzi siniflərdə məlumat azdırsa (məsələn, tibbi şəkillərdə nadir xəstəlik), CVAE ilə həmin sinif üçün süni, lakin real görünən nümunələr yaradıb təlim verisini artırırsınız. Classifier sonra daha yaxşı öyrənir.

**3. Üslubu sinifdən ayırmaq (disentanglement)**
Sizin 9×9 şəbəkə təcrübəsi məhz bunu yoxlayır: latent fəza yalnız üslubu (qalınlıq, əyiklik) saxlayır, sinif isə label-dən gəlir. Bu, modelin nəyi öyrəndiyini anlamaq üçün vacibdir.

**4. Üslub transferi**
Bir şəkildən latent kod (üslub) çıxarırsınız, label-i dəyişib decoder-ə verirsiniz: eyni əl yazısı üslubunda başqa rəqəm alınır. Məsələn, "mənim əl yazımla 2-ni 8 et".

**5. Real tətbiqlər**
Mətndən/şərtdən şəkil generasiyası (şərti generasiya) məhz bu ideyanın böyüdülmüş formasıdır: şərt (label, mətn) verilir, model məzmun yaradır. Dərman molekulu dizaynı, musiqi generasiyası, sintetik verilər də belədir.

## Sizin təcrübənin məqsədi

Bu konkret kod praktik tətbiq deyil, **modelin düzgün öyrənib-öyrənmədiyini yoxlamaq üçün vizual testdir**. Əgər label dəyişəndə rəqəm dəyişir, latent nöqtə dəyişəndə yalnız üslub dəyişirsə, CVAE işləyir deməkdir.

Qısası: decoder-in işi yeni məlumat **yaratmaqdır**, tanımaq yox. Label-i özümüz veririk, çünki "nə yaratmaq istəyirik" sualının cavabı bizdədir.